In [ ]:
# ============ IMPORTS ============
import os
import sys
import json
import uuid
import sqlite3
from datetime import datetime
from typing import Annotated, Literal
from typing_extensions import TypedDict

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.tools import tool
from langchain_core.runnables import RunnableConfig
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.sqlite import SqliteSaver

# Local helpers
sys.path.append(os.path.abspath("../../.."))
from helpers import get_llm

from dotenv import load_dotenv
load_dotenv()

print("Imports OK")

In [ ]:
# ============ LLM INITIALIZATION ============
# Uses platform defaults: Databricks on macOS, Groq on Windows.
llm = get_llm()

In [ ]:
from typing_extensions import TypedDict, Annotated
from langchain_core.messages import AnyMessage
from operator import add

class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], add]

In [ ]:
from langchain_core.messages import SystemMessage

def chat_llm_node(state: MessagesState):
    history = [
        SystemMessage(content="You are a customer support assistant.")
    ]
    history.extend(state["messages"])

    reply = llm.invoke(history)       

    return {"messages": [reply]}

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

builder = StateGraph(MessagesState)
builder.add_node("chat_llm", chat_llm_node)
builder.add_edge(START, "chat_llm")
builder.add_edge("chat_llm", END)

graph = builder.compile(checkpointer=checkpointer)

In [ ]:
import time

def print_turn(state_in: dict, response: dict, latency: float) -> None:
    """Pretty-print the input human message and the final AI reply, then latency + token usage."""
    state_in["messages"][-1].pretty_print()
    ai_msg = response["messages"][-1]
    ai_msg.pretty_print()

    tokens = 0
    usage = getattr(ai_msg, "usage_metadata", None)
    if usage:
        tokens = usage.get("total_tokens", 0)
    else:
        meta = getattr(ai_msg, "response_metadata", {}) or {}
        token_usage = meta.get("token_usage") or meta.get("usage") or {}
        tokens = token_usage.get("total_tokens", 0)

    # ANSI: cyan for latency, green for tokens
    print(f"\n\033[96mLatency: {latency:.2f}s\033[0m")
    print(f"\033[92mTotal tokens: {tokens}\033[0m")

In [ ]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "ticket-seq"}}

state1 = {
    "messages": [
        HumanMessage(content=(
            "Hi, I'm being charged twice for my subscription. "
            "Can you help me figure out what's going on?"
        ))
    ]
}
_t = time.time()
response1 = graph.invoke(state1, config=config)
lat1 = time.time() - _t

In [ ]:
print_turn(state1, response1, lat1)

In [ ]:
from langchain_core.messages import HumanMessage

state2 = {
    "messages": [
        HumanMessage(content=(
            "I think this started after I changed "
            "my billing address last month."
        ))
    ]
}
_t = time.time()
response2 = graph.invoke(state2, config=config)
lat2 = time.time() - _t

In [ ]:
print_turn(state2, response2, lat2)

In [ ]:
from langchain_core.messages import HumanMessage

state3 = {
    "messages": [
        HumanMessage(content=(
            "Can you summarize what we did just now?"
        ))
    ]
}
_t = time.time()
response3 = graph.invoke(state3, config=config)
lat3 = time.time() - _t

In [ ]:
print_turn(state3, response3, lat3)

### Performance
LangGraph’s state snapshot includes the full state and metadata for the latest checkpoint along with the token usage and the response times.

From the performance metrics it is clear that the number of prompt tokens grows roughly linearly with the number of turns. This also leads to growing costs, because you pay for all past tokens on every call to the LLM.

The total response time per turn tends to increase as prompts get larger directly affecting latency.

In short conversations, this works fine and ensures no detail is lost. For many internal tools and early-stage prototypes, this may be completely acceptable.

However, as conversations grow longer, this approach runs into trouble. As we discussed in Part 15, simply throwing everything into the prompt is not a scalable strategy:

Every extra token has a cost. Sending a massive, unmanaged history on every turn quickly becomes financially unsustainable.
Large prompts lead to high inference latency. Waiting for longer times per answer is unacceptable in most production environments.
Sequential memory is, therefore, best treated as a baseline as it gives us the most straightforward behavior and is easy to instrument. But it does not scale well.

With this understanding, we should start improving this behavior by introducing techniques that keep the spirit of sequential memory (don’t lose what matters) while reducing cost, latency, and context bloat.

So let's go ahead and take a look at the sliding window approach.

### Sliding window memory


With sequential memory, we kept every single turn in the conversation and sent the entire history to the LLM on each request. This gave us perfect recall inside a single conversation, but the prompt kept growing with every turn, along with cost and latency.

The next step is to keep a hard bound on how much context the agent actually sees from the full conversation history.

That’s what sliding window memory does.

In this approach, instead of retaining the entire conversation history, the agent keeps only the most recent N messages as context. As new messages arrive, the oldest ones are dropped, and the window slides forward.

Intuitively, the flow now looks like this:

The user starts the conversation with the AI agent taking turns.
Each turn is added to the conversation history.
Before generating a new response, we keep only the last few turns (for example, the last 6 messages or the last 1,000 tokens).
This trimmed window, plus the new user query, is sent to the LLM.
Everything outside the window is effectively forgotten by the agent for this turn.

You can think of it as the agent having a short-term memory buffer. It remembers what just happened, but after a while, starts to forget the earliest parts of the conversation.

This is the first step towards effective memory management.

In [ ]:
from typing_extensions import TypedDict, Annotated
from langchain_core.messages import AnyMessage
from operator import add
from langgraph.graph.message import add_messages
from langchain_core.messages import RemoveMessage

class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

In [ ]:
MAX_MESSAGES = 4

def truncate_messages_node(state):
    msgs = state["messages"]
    print(f"[truncate] incoming len = {len(msgs)}")   # <-- add here

    if len(msgs) <= MAX_MESSAGES:
        print("[truncate] no-op (under limit)")
        return {}
    to_drop = msgs[:-MAX_MESSAGES]

    return {"messages": [RemoveMessage(id=m.id) for m in to_drop]}

In [ ]:
from langchain_core.messages import SystemMessage

def chat_llm_node(state: MessagesState):
    history = [
        SystemMessage(content="You are a customer support assistant.")
    ]
    print(f"[chat_llm] LLM sees {len(state['messages'])} messages")
    print(f"Current history length: {len(history)}")
    print("Current message state:")
    for m in state["messages"]:
        m.pretty_print()
    history.extend(state["messages"])

    reply = llm.invoke(history)       

    return {"messages": [reply]}

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

builder = StateGraph(MessagesState)

builder.add_node("truncate", truncate_messages_node)
builder.add_node("chat_llm", chat_llm_node)

# flow: START -> truncate -> chat_llm -> END
builder.add_edge(START, "truncate")
builder.add_edge("truncate", "chat_llm")
builder.add_edge("chat_llm", END)

sliding_graph = builder.compile(checkpointer=checkpointer)

In [ ]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "ticket-sliding"}}

state1 = {
    "messages": [
        HumanMessage(content=(
            "Hi, I'm being charged twice for my subscription. "
            "Can you help me figure out what's going on? "
            "My favorite color is blue by the way."
        ))
    ]
}
response1 = sliding_graph.invoke(state1, config=config)

In [ ]:
print_turn(state1, response1, lat1)

In [ ]:
from langchain_core.messages import HumanMessage

state2 = {
    "messages": [
        HumanMessage(content=(
            "I think this started after I changed "
            "my billing address last month."
        ))
    ]
}
response2 = sliding_graph.invoke(state2, config=config)

In [ ]:
print_turn(state2, response2, lat2)

In [ ]:
from langchain_core.messages import HumanMessage

state3 = {
    "messages": [
        HumanMessage(content=(
            "What is my favorite color?"
        ))
    ]
}
response3 = sliding_graph.invoke(state3, config=config)

In [ ]:
print_turn(state3, response3, lat3)

### Performance
With sequential memory, total tokens per turn keep growing more or less linearly with the number of turns. With a sliding window, tokens per turn grow only until the window is full. After that, the total token count stabilizes, because older messages are being dropped as new ones come in.

Latency follows a similar pattern. In sequential we saw gradually increasing response times whereas with sliding window response times grow early, then plateau once the prompt size stabilizes

Sliding window memory is much more efficient than sequential memory, but it comes with a clear limitation, i.e., anything that falls outside the window is wiped off from the agent's memory.

In a customer support agent setup, this can be problematic when:

A critical piece of information was mentioned early (e.g. “I’m on the Enterprise plan” or “this only happens on Firefox”) and has since scrolled out of the window.
The user refers back to something that happened many turns ago (“remember the error I got yesterday?”) and that message is no longer present in the truncated context.
Support tickets are long-running and interleave multiple threads of discussion.
In short, sliding window keeps cost and latency under control. It works well for relatively short, focused conversations where the most important information is in the last few messages. But it has no way of remembering older, relevant details once they fall outside the window.

This is where our next technique, summarization, comes in.

### Summarization-based memory
If you’ve used AI-powered coding editors like Cursor or Claude Code, you’ve already seen summarization-based memory in action.

As you chat with the assistant, ask it to refactor files, review diffs, and explain architecture decisions, there comes a point where the context window starts to fill up. Instead of failing or blindly truncating, these tools quietly start to:

compress older parts of the conversation into short summaries
retain key decisions, constraints, and naming conventions
keep only the most recent turns in full detail

As a result, the assistant still remembers the important stuff, ensuring that the actual prompt stays within a manageable size to fit into the LLM's context window.

That’s exactly the idea behind summarization-based memory.

Instead of dropping old information entirely, the agent could remember it in a condensed form. The idea is to regularly take the conversation so far, create a brief summary of the important points, and use that summary as a substitute for the full history.

In [ ]:
from typing_extensions import TypedDict, Annotated
from langchain_core.messages import AnyMessage
from operator import add
from langgraph.graph.message import add_messages
from langchain_core.messages import SystemMessage, HumanMessage, RemoveMessage


class SummarizationState(TypedDict):
    summary: str
    
    buffer: Annotated[list[AnyMessage], add_messages]

In [ ]:
import time

def print_turn(state_in: dict, response: dict, latency: float) -> None:
    """Pretty-print the input human message and the final AI reply, then latency + token usage."""
    state_in["buffer"][-1].pretty_print()
    ai_msg = response["buffer"][-1]
    ai_msg.pretty_print()

    tokens = 0
    usage = getattr(ai_msg, "usage_metadata", None)
    if usage:
        tokens = usage.get("total_tokens", 0)
    else:
        meta = getattr(ai_msg, "response_metadata", {}) or {}
        token_usage = meta.get("token_usage") or meta.get("usage") or {}
        tokens = token_usage.get("total_tokens", 0)

    # ANSI: cyan for latency, green for tokens
    print(f"\n\033[96mLatency: {latency:.2f}s\033[0m")
    print(f"\033[92mTotal tokens: {tokens}\033[0m")

In [ ]:
MESSAGE_THRESHOLD = 4

def summarize_buffer_node(state: SummarizationState):
    buffer = state["buffer"]
    current_summary = state.get("summary", "")

    if len(buffer) < MESSAGE_THRESHOLD:
        return {}

    lines = []
    for i, m in enumerate(buffer):
        role = "User" if i % 2 == 0 else "Assistant"
        lines.append(f"{role}: {m.content}")
    buffer_text = "\n".join(lines)

    # system = [
    #     SystemMessage(
    #         content=(
    #             "Maintain summary of a support conversation.\n"
    #             "Keep only important facts and decisions.\n"
    #             "Prefer short bullet points when possible."
    #         )
    #     )
    # ]

    system = SystemMessage(content=(
        "Maintain a summary of a support conversation.\n"
        "Keep only important facts and decisions.\n"
        "Prefer short bullet points when possible."
    ))

    user_content = (
        f"Previous summary:\n{current_summary or '(none)'}\n\n"
        f"New conversation turns:\n{buffer_text}\n\n"
        "Produce an updated summary."
    )


    summary_reply = llm.invoke([system, HumanMessage(content=user_content)])

    new_summary = summary_reply.content

    return {
        "summary": new_summary,
        "buffer": [RemoveMessage(id=m.id) for m in buffer] 
    }

In [ ]:
def chat_llm_with_summary_node(state: SummarizationState):
    system_prompt = (
        "You are a helpful customer support assistant. "
        "Use the conversation summary for global context "
        "and the recent messages for local detail."
    )

    summary = state.get("summary", "")
    buffer = state["buffer"]

    messages = [SystemMessage(content=system_prompt)]

    if summary:
        messages.append(
            SystemMessage(
                content=f"Conversation summary so far:\n{summary}"
            )
        )

    messages.extend(buffer)

    reply = llm.invoke(messages)
    return {"buffer": [reply]}

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

builder = StateGraph(SummarizationState)

builder.add_node("summarize", summarize_buffer_node)
builder.add_node("chat_llm", chat_llm_with_summary_node)

# flow: START -> summarize -> chat_llm -> END
builder.add_edge(START, "summarize")
builder.add_edge("summarize", "chat_llm")
builder.add_edge("chat_llm", END)

summary_graph = builder.compile(checkpointer=checkpointer)

In [ ]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "ticket-summarize"}}

state1 = {
    "summary": "",
    "buffer": [
        HumanMessage(
            content=(
                "Hi, my Team Analytics workspace isn't updating our "
                "weekly reports. The data is stuck on last Monday."
            )
        )
    ],
}
response1 = summary_graph.invoke(state1, config=config)

In [ ]:
print_turn(state1, response1, lat1)

In [ ]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "ticket-summarize"}}

state2 = {
    "summary": "",
    "buffer": [
        HumanMessage(
            content=(
                "We use the Growth plan with 15 seats. "
                "The workspace name is 'Marketing Performance Q4'."
            )
        )
    ],
}
response2 = summary_graph.invoke(state2, config=config)

In [ ]:
print_turn(state2, response2, lat2)

In [ ]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "ticket-summarize"}}

state3 = {
    "summary": "",
    "buffer": [
        HumanMessage(
            content=(
                "Can you remind me our workspace name "
                "and the problem we are facing?"
            )
        )
    ],
}
response3 = summary_graph.invoke(state3, config=config)

In [ ]:
print_turn(state3, response3, lat3)